In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split, KFold


In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
# Read the dataset Q3_data.csv using read_csv()
csv_file_path = os.path.join(path, 'Q3_data.csv')

df = pd.read_csv(csv_file_path)

In [ ]:
# Task 2: Write your code here:
# Inspect the first few rows using head()
df.head()


In [ ]:
# Task 3: Write your code here:
# Display dataset information using info()
df.info()
df.shape # 20001 row!

In [ ]:
# Task 4: Write your code here:
# Show statistical description using describe()
df.describe()

In [ ]:
df.isnull().sum()


In [ ]:
# Task 1: Write your code here:
df.isnull().sum()

# df['D_142']/df.shape[0]
# Analyze missing values
missing_percentage = (df.isnull().sum() / len(df)) * 100
missing_data = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Percentage': missing_percentage.values,
    'column_type' : missing_percentage.dtype
})
missing_data # 184	D_142	84.285786	float64 has 84% of missing values so drop cuz it is a lot
df = df.drop(columns=['D_142'])
df.shape
df['P_2'] = df['P_2'].fillna(df['P_2'].mean())
df['B_2'] = df['B_2'].fillna(df['B_2'].mean())
df['D_141'] = df['D_141'].fillna(df['D_141'].mean())
df['D_143'] = df['D_143'].fillna(df['D_143'].mean())
df['D_144'] = df['D_144'].fillna(df['D_144'].mean())
df['D_145'] = df['D_145'].fillna(df['D_145'].mean())


In [ ]:
df.isnull().sum()


In [ ]:
# Task 2: Write your code here:
# Check and remove duplicates if any exist
# Task 3: Write your code here:
# Check and remove duplicates if any exist
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 3: Write your code here:
# Encode categorical variables if needed
# df.dtypes are all numerical so no need skip this stip
df.info() #dtypes: float64(147), int64(41)


In [ ]:
# Task 4: Write your code here:
# Apply feature scaling to numerical features (Use StandardScaler)
X = df.drop('Target', axis=1)
y = df['Target']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled


In [ ]:
# Task 5: Write your code here:
# Check for target imbalance and state if it is imbalanced or not
import seaborn as sns
print("Target Distribution:")
print(y.value_counts(normalize=True))
sns.countplot(x=y)
plt.title("Target Distribution")
plt.show()

 ## imbalance class 1 is minor so need stratify = y :)
#   Target
# 0    0.736563
# 1    0.263437


In [ ]:
# Task 1: Write your code here:
## did it before scaling :) avoid data leakage

In [ ]:
from IPython.display import clear_output

%pip install kagglehub catboost xgboost tqdm -q

clear_output()

In [ ]:
# Task 2,3,4,5: Write your code here:
# Rule of thumb: If class distributions are not equal, then our data is imbalanced. Use StratifiedKFold and focus on F1-score. And if data distribution is balanced, StratifiedKFold will act like regular KFold, so always use StratifiedKFold :)
from sklearn.model_selection import StratifiedKFold
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
# Train
acc = []
f1 = []
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
model = CatBoostClassifier(
      verbose=0,
      n_estimators=320,
      max_depth=4
  )

for fold_idx, (train_index, test_index) in enumerate(skf.split(X_scaled, y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  # 1. Split data
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # 2. Train & Validate sklearn models
  model.fit(X_train, y_train) # train
  y_pred = model.predict(X_test) # validate


    # 3. Save metrics for that model in this
  f1_fold = f1_score(y_test, y_pred, zero_division=0)
  accuracy = accuracy_score(y_test, y_pred)

  f1.append(f1_fold)
  acc.append(accuracy)

    # accuracy = accuracy_score(y_test, y_pred)
    # precision = precision_score(y_test, y_pred, zero_division=0)
    # recall = recall_score(y_test, y_pred, zero_division=0)
    # f1 = f1_score(y_test, y_pred, zero_division=0)

    # all_results[model_name]['accuracy'].append(accuracy)
    # all_results[model_name]['precision'].append(precision)
    # all_results[model_name]['recall'].append(recall)
    # all_results[model_name]['f1'].append(f1)

In [ ]:
import numpy as np
avg_f1 = np.mean(f1, axis=0)
avg_acc = np.mean(acc, axis=0)
avg_f1, avg_acc

In [ ]:
# Task 1: Write your code here:
# Task 1: Write your code here:
# Plot feature importance from your trained model
# Gather importances from the models (from the last fold)
importances = {}

importances['CatBoostClassifier'] = model.feature_importances_

# Create a 1x3 plot
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
axes = axes.flatten()
features = X.columns

for i, (model_name, imp) in enumerate(importances.items()):
  # Sort features by importance for a cleaner plot
  sorted_idx = np.argsort(imp)

  ax = axes[i]
  ax.barh(features[sorted_idx], imp[sorted_idx])
  ax.set_title(f"{model_name} Feature Importance")
  ax.set_xlabel("Importance Score")

plt.tight_layout()
plt.show()

In [ ]:
(features[sorted_idx], imp[sorted_idx])

In [ ]:
# feature_importance = pd.DataFrame({
#     'feature': feature_cols,
#     'importance': model.feature_importances_
# }).sort_values('importance', ascending=False)

In [ ]:
# importances['CatBoostClassifier'].sort_values('importance', ascending=False)

In [ ]:
# # Task 2: Write your code here:
# # Identify and print the name of the most important feature (the 'golden feature')
# importances['CatBoostClassifier'] = model.feature_importances_
# importances['CatBoostClassifier'].max

(features[sorted_idx], imp[sorted_idx])
# SOLUTION IS R_4

In [ ]:
# Task Bonus: Write your code here: